# Short-window P picker: 60 s → 30/20/10/5 s

**EQViT-torch Phase 2** — reproducible, event-disjoint, latency-aware workflow.

> Research prototype: validate locally before operational EEW use.

## Research question

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from torch.utils.data import DataLoader
from eqvit_torch import *
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
H5=Path('/path/to/TXED_20231111.h5'); IDS=Path('/path/to/ID_20231111.npy')
SEED=2026; np.random.seed(SEED); torch.manual_seed(SEED)
print('device:',DEVICE)

In [ ]:
grid=pd.DataFrame({'samples':[6000,3000,2000,1000,500]}); grid['seconds']=grid.samples/100; display(grid)

In [ ]:
def crop_random_p(x,p,n,rng,min_pre_frac=.1,max_pre_frac=.8):
 pre=int(rng.uniform(min_pre_frac,max_pre_frac)*n); start=p-pre; out=np.zeros((n,3),np.float32); s=max(0,start); e=min(len(x),start+n); out[s-start:e-start]=x[s:e]; return out,p-start
# P location is randomized. Never center P systematically in short-picker testing.

In [ ]:
models={n:EQViTPicker(samples=n,patch=40) for n in [6000,3000,2000,1000]}; models[500]=EQViTPicker(samples=500,patch=20)
for n,m in models.items(): print(n,sum(p.numel() for p in m.parameters()))

In [ ]:
metrics=['P MAE (s)','recall @0.2 s','recall @0.5 s','false alarms/hour','M>=3 recall','M>=4 recall','CPU latency','GPU latency']
print(pd.DataFrame(index=grid.seconds,columns=metrics))

## Continuous-data stress test